In [ ]:
%load_ext autoreload
%autoreload 2


# MethylSeg Components Example

This notebook shows how to use the loaded `assigner`, `analyzer`, and `segmentor` objects directly after loading a pretrained pathway model.


In [ ]:
from pathlib import Path
import pandas as pd

from methylseg import (
    MethylDataPrep,
    MethylSegPathway,
    MethylStateAssignmentMethod,
    MethylationStates,
)


In [ ]:
ip = get_ipython()
if "__vsc_ipynb_file__" in ip.user_ns:
    EXAMPLES_DIR = Path(ip.user_ns["__vsc_ipynb_file__"]).resolve().parent
else:
    EXAMPLES_DIR = Path.cwd()

PACKAGE_ROOT = EXAMPLES_DIR.parent
REFERENCE_DIR = PACKAGE_ROOT / "data" / "reference_files"

OUT_DIR = EXAMPLES_DIR / "out" / "methylseg_components_example_output"


In [ ]:
model = MethylSegPathway.get_pretrained_model(OUT_DIR, resolution="450k")


In [ ]:
sample_info, sample_info_removed = MethylDataPrep(
    meth_file=REFERENCE_DIR / "TCGA-BD-A3EP-01A_450k.tsv.gz",
    sample_id="TCGA-BD-A3EP-01A",
    resolution="450k",
    remove_low_coverage_like_cpgs=True,
).prepare()

sample_info.sample_id


## Assigner

Apply the trained KMeans model directly to a new sample and inspect the engineered emissions.


In [ ]:
assigner = model.assigner

meth_data, emission_df, pca_scores, raw_distances, raw_labels, relabeled_labels = assigner.apply_kmeans_to_sample(
    sample_info=sample_info,
    chrom="chr1",
)

pd.DataFrame({
    "CpG_beg": meth_data["CpG_beg"],
    "beta": meth_data["beta"],
    "kmeans_state": relabeled_labels,
}).head()


In [ ]:
fig = assigner.plot_embedding(
    emission_df=emission_df,
    labels=relabeled_labels,
    meth_data=meth_data,
    sample_info=sample_info,
    chrom="chr1",
    method="pca",
    include_biplot=True,
    include_metrics=True,
    label_title="KMeans state",
    hexbin=False,
)


## Analyzer

Use the analyzer to inspect the learned KMeans states and the rule-based labels derived from them.


In [ ]:
analyzer = model.analyzer

analyzer.pretty_print_rules()


In [ ]:
rule_based_labels = analyzer.define_states_by_rules(
    sample_info=sample_info,
    chrom="chr1",
    sample_emissions=emission_df,
)

pd.Series(rule_based_labels).value_counts()


In [ ]:
analyzer.plot_feature_distributions_by_kmeans_state()


In [ ]:
fig = analyzer.plot_labels(
    sample_info=sample_info,
    sample_info_removed=sample_info_removed,
    chrom="chr1",
    label_source="rule_based",
    label_title="Rule-based state",
)


In [ ]:
analyzer.evaluate_clustering_concordance(
    sample_info=sample_info,
    chrom="chr1",
)


## Segmentor

Segment the sample using the current state assignment method and inspect the resulting HMM regions.


In [ ]:
segmentor = model.segmentor

segmented_meth, _ = segmentor.segment_sample(sample_info=sample_info, chrom="chr1")
segmented_meth[["CpG_chrm", "CpG_beg", "state_readable", "hmm_state_readable"]].head()


In [ ]:
regions_chr1 = segmentor.create_regions()
regions_chr1.head()


In [ ]:
clean_summary_paths, clean_dir = model.get_clean_regions(
    regions_df=regions_chr1,
    sample_id=sample_info.sample_id,
    chrom="chr1",
)
clean_summary_paths, clean_dir


In [ ]:
fig = model.plot_labels(
    label_source="hmm",
    sample_info=sample_info,
    sample_info_removed=sample_info_removed,
    chrom="chr1",
    use_cleaned_regions=True,
    overlay_state="PMD",
    label_title="HMM state",
)
